# 实验报告03：正则表达式与网络结构化数据

作者：罗文希；学号：20244081107。

# 一、实验环境与提交要求
## 1.1 第一个代码框：身份与环境记录

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244081107

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] ALICE
[学号]       20244081107
[机器指纹] c13bd608a3df89c7
[RUN_ID]   20244081107-c13bd608a3df89c7
[Python]    3.12.15


解题说明：以下使用 Python (ALICE) 内核，工作目录为 EXP03。

In [2]:
import sys
from pathlib import Path
print("解释器：", sys.executable)
print("工作目录：", Path.cwd())
assert Path(sys.executable).parent.name == "ALICE"
assert CONDA_ENV == "ALICE"

解释器： Z:\Anaconda\anaconda\envs\ALICE\python.exe
工作目录： L:\26年秋季\Python_Data_Processing\EXP03


# 二、实验内容与过程

## 实验1：正则表达式的基础匹配

在本实验开始时导入 `re`。操作任务：

### 2.1.1

在 `"cat on mat; later: cats and dogs"` 上使用正则表达式，分别匹配文本中的数字 `\d+`、`\s+` 和 `\bcat\b`。

In [3]:
import re
text = "cat on mat; later: cats and dogs"
for pattern in [r"\d+", r"\s+", r"\bcat\b"]:
    print(pattern, re.findall(pattern, text))
assert re.findall(r"\d+", text) == []
assert re.findall(r"\bcat\b", text) == ["cat"]

\d+ []
\s+ [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b ['cat']


**结果说明：** 原文本没有数字，所以 `\d+` 返回空列表；`\s+` 匹配连续空白；`\bcat\b` 只匹配完整单词 cat，不匹配 cats 中的子串。

### 2.1.2

在 `"cat dog bird CAT"` 上使用 `cat|dog`。用 `re.findall(r"a|aa|aaa", "aaaa")` 和 `re.findall(r"aaa|aa|a", "aaaa")` 展示差异。

In [4]:
print("cat|dog:", re.findall(r"cat|dog", "cat dog bird CAT"))
short_first = re.findall(r"a|aa|aaa", "aaaa")
long_first = re.findall(r"aaa|aa|a", "aaaa")
print("短分支在前：", short_first)
print("长分支在前：", long_first)
assert short_first == ["a"] * 4
assert long_first == ["aaa", "a"]

cat|dog: ['cat', 'dog']
短分支在前： ['a', 'a', 'a', 'a']
长分支在前： ['aaa', 'a']


**结果说明：** 默认区分大小写，所以 CAT 不匹配。分支按从左到右的顺序尝试，首个成功分支即被接受；因此两种顺序分别得到四个 a 和 aaa、a。

### 2.1.3

在 `"ct cat caat caaat"` 上分别使用 `ca*t`、`ca+t`、`ca?t`、`ca{2}t` 和 `ca{1,2}t`，输出匹配列表并说明各量词的含义。

In [5]:
text = "ct cat caat caaat"
for pattern in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(pattern, re.findall(pattern, text))

ca*t ['ct', 'cat', 'caat', 'caaat']
ca+t ['cat', 'caat', 'caaat']
ca?t ['ct', 'cat']
ca{2}t ['caat']
ca{1,2}t ['cat', 'caat']


**结果说明：** 量词均作用于前面的 a：`*` 表示零次或多次，`+` 表示一次或多次，`?` 表示零次或一次，`{2}` 表示恰好两次，`{1,2}` 表示一次或两次。末尾 t 必须紧接这些 a，因此不会把 caaat 的局部误当作 caat。

## 实验2：分组、回溯引用与替换

操作任务：

### 2.2.1

在 `"联系：alice@example.com；备选：bob@test.org"` 中，用捕获组提取每个邮箱地址的用户名和域名。

In [6]:
text = "联系：alice@example.com；备选：bob@test.org"
addresses = re.findall(r"([\w.+-]+)@([\w.-]+\.[A-Za-z]+)", text)
for username, domain in addresses:
    print("用户名：", username, "域名：", domain)
assert addresses == [("alice", "example.com"), ("bob", "test.org")]

用户名： alice 域名： example.com
用户名： bob 域名： test.org


**结果说明：** 两个捕获组分别匹配 @ 前的用户名与 @ 后的域名；该表达式针对本题示例，不是完整的邮箱地址验证器。

### 2.2.2

在 `"cat cat, dog cat, bird bird"` 中，用后向引用找出连续重复的单词。

In [7]:
text = "cat cat, dog cat, bird bird"
pattern = r"\b(\w+)\s+\1\b"
pairs = [match.group(0) for match in re.finditer(pattern, text)]
print("连续重复：", pairs)
assert pairs == ["cat cat", "bird bird"]

连续重复： ['cat cat', 'bird bird']


**结果说明：** `\1` 要求第二个单词与第一捕获组相同；`\s+` 要求中间只有空白，因此 dog cat 不符合。

### 2.2.3

将 `"2024-05-01\n2024/05/02\n2024.05.03"` 中的日期从“年-月-日”形式转换为“月-日-年”形式，保留原来的行数。可用 `re.sub()`、捕获组和替换表达式完成；展示转换前后文本。

In [8]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
converted = re.sub(r"\b(\d{4})[-/.](\d{2})[-/.](\d{2})\b", r"\2-\3-\1", dates)
print("转换前：\n" + dates)
print("转换后：\n" + converted)
assert converted == "05-01-2024\n05-02-2024\n05-03-2024"
assert len(converted.splitlines()) == len(dates.splitlines())

转换前：
2024-05-01
2024/05/02
2024.05.03
转换后：
05-01-2024
05-02-2024
05-03-2024


**结果说明：** 三种日期分隔符均被识别，替换只调整每行的年月日顺序并统一为连字符，换行符保留。

### 2.2.4

将 `"联系我：123-456-7890 或 987.654.3210"` 中的电话号码统一为 `(123) 456-7890` 形式。

In [9]:
phones = "联系我：123-456-7890 或 987.654.3210"
formatted = re.sub(r"\b(\d{3})[-.](\d{3})[-.](\d{4})\b", r"(\1) \2-\3", phones)
print(formatted)
assert formatted == "联系我：(123) 456-7890 或 (987) 654-3210"

联系我：(123) 456-7890 或 (987) 654-3210


**结果说明：** 分别捕获三段数字，保留每个号码自身的数字，只统一括号、空格与连字符。

## 实验3：使用 BeautifulSoup 解析 HTML

使用以下字符串，不依赖网页下载：

In [10]:
html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

### 2.3.1

用 `BeautifulSoup(html_doc, "html.parser")` 构建解析树，输出页面标题的文本、h1 的标签名与文本。

In [11]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html_doc, "html.parser")
print("标题：", soup.title.get_text())
print("h1 标签名：", soup.h1.name)
print("h1 文本：", soup.h1.get_text())
assert soup.title.get_text() == "实验课程"
assert soup.h1.name == "h1" and soup.h1.get_text() == "数据分析"

标题： 实验课程
h1 标签名： h1
h1 文本： 数据分析


**结果说明：** 使用本实验公共字符串构建解析树，标签的 name 和文本内容分别通过属性与 get_text() 获取。

### 2.3.2

使用 `find_all()` 找出所有 `class="course"` 的段落和所有链接；逐项输出链接文字与 `href` 属性。

In [12]:
paragraphs = soup.find_all("p", class_="course")
links = soup.find_all("a")
for paragraph in paragraphs:
    print("课程段落：", paragraph.get_text())
for link in links:
    print("链接文字：", link.get_text(), "href：", link.get("href"))
assert len(paragraphs) == 2 and len(links) == 2

课程段落： 第六讲：正则表达式
课程段落： 第七讲：结构化数据
链接文字： 讲义六 href： https://example.org/lecture06
链接文字： 讲义七 href： https://example.org/lecture07


**结果说明：** 复用 2.3.1 的 soup。class_ 对应 HTML 的 class 属性，find_all 返回全部符合条件的标签。

### 2.3.3

用 `get_text()` 提取页面可见文本；比较整个文档、单个段落和单个链接的文本结果。

In [13]:
print("整个文档：\n" + soup.get_text(separator="\n", strip=True))
print("单个段落：", soup.find("p").get_text())
print("单个链接：", soup.find("a").get_text())
print("body 文本：\n" + soup.body.get_text(separator="\n", strip=True))

整个文档：
实验课程
数据分析
第六讲：正则表达式
第七讲：结构化数据
讲义六
讲义七
单个段落： 第六讲：正则表达式
单个链接： 讲义六
body 文本：
数据分析
第六讲：正则表达式
第七讲：结构化数据
讲义六
讲义七


**结果说明：** 复用 2.3.1 的 soup。整个文档的 get_text() 提取所有后代文本，也包含 head 中的标题“实验课程”；单个段落仅有“第六讲：正则表达式”，单个链接仅有“讲义六”，不包含 href。body 的文本更接近页面主体可见内容。get_text() 不执行浏览器的 CSS 可见性判断，separator 和 strip 用于整理文本分隔与首尾空白。